# Pag-book ng Hotel gamit ang Priority Member Middleware

Ipinapakita ng notebook na ito ang **function-based middleware** gamit ang Microsoft Agent Framework. Pinagyayaman namin ang halimbawa ng conditional workflow sa pamamagitan ng pagdaragdag ng middleware layer na nagbibigay ng espesyal na pribilehiyo sa mga priority members.

## Mga Matututuhan Mo:
1. **Function-Based Middleware**: Salain at baguhin ang mga resulta ng function
2. **Pag-access sa Context**: Basahin at baguhin ang `context.result` pagkatapos ng pagpapatupad
3. **Pagpapatupad ng Business Logic**: Mga benepisyo para sa priority member
4. **Pag-override ng Resulta**: Baguhin ang mga kinalabasan ng function base sa status ng user
5. **Parehong Workflow, Ibang Resulta**: Mga pagbabagong dulot ng middleware sa pag-uugali

## Arkitektura ng Workflow kasama ang Middleware:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## Pangunahing Pagkakaiba mula sa Conditional Workflow:

**Walang Middleware** (14-conditional-workflow.ipynb):
- Walang kuwarto sa Paris → I-route sa alternative_agent

**May Middleware** (notebook na ito):
- Regular na user + Paris → Walang kuwarto → I-route sa alternative_agent
- Priority user + Paris → 🌟 Middleware ang nag-override! → May kuwarto → I-route sa booking_agent

## Mga Kinakailangan:
- Nakainstall ang Microsoft Agent Framework
- Pag-unawa sa conditional workflows (tingnan ang 14-conditional-workflow.ipynb)
- GitHub token o OpenAI API key
- Pangunahing pag-unawa sa mga middleware pattern


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Hakbang 1: Tukuyin ang Mga Pydantic Model para sa Mga Istrakturadong Output

Itong mga modelo ay tumutukoy sa **schema** na ibabalik ng mga ahente. Nagdagdag kami ng isang `priority_override` na patlang upang subaybayan kung kailan binabago ng middleware ang resulta ng availability.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Step 2: Tukuyin ang Database ng Mga Miyembrong May Prayoridad

Para sa demo na ito, gagawa tayo ng simulasyon ng database ng mga miyembrong may prayoridad. Sa produksyon, ito ay magtatanong sa isang tunay na database o API.

**Mga Miyembrong May Prayoridad:**
- `alice@example.com` - VIP na miyembro
- `bob@example.com` - Premium na miyembro  
- `priority_user` - Test account


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## Hakbang 3: Gumawa ng Kasangkapang Pang-book ng Hotel

Katulad ng conditional workflow, ngunit ngayon ito ay maa-intercept ng middleware!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Step 4: 🌟 Gumawa ng Priority Check Middleware (ANG PANGUNAHING TAMPOK!)

Ito ang **pangunahin na functionality** ng notebook na ito. Ang middleware ay:

1. **Humaharang** sa tawag ng hotel_booking function
2. **Isinasagawa** ang function nang normal sa pamamagitan ng pagtawag ng `next(context)`
3. **Sinusuri** ang resulta sa `context.result`
4. **Pinapalitan** ang resulta kung priority ang user at walang bakanteng kwarto
5. **Ibinabalik** ang binagong resulta pabalik sa agent

**Pangunahing Pattern:**
```python
async def my_middleware(context, next):
    await next(context)  # Isagawa ang function
    # Ngayon ang context.result ay naglalaman ng output ng function
    if some_condition:
        context.result = new_value  # Palitan!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## Hakbang 5: Tukuyin ang mga Function ng Kondisyon para sa Routing

Parehong mga function ng kondisyon tulad ng sa conditional workflow - sinusuri nila ang nakaayos na output upang matukoy ang routing.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## Step 6: Gumawa ng Custom Display Executor

Parehong executor tulad ng dati - nagpapakita ng huling output ng workflow.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## Hakbang 7: I-load ang Mga Variable ng Kapaligiran

I-configure ang LLM client (GitHub Models o OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Hakbang 8: Gumawa ng AI Agents gamit ang Middleware

**PANGUNAHING PAGKAKAIBA:** Kapag lumilikha ng availability_agent, ipinapasa natin ang parameter na `middleware`!

Ganito natin ini-inject ang priority_check_middleware sa pipeline ng pagtawag sa function ng agent.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Hakbang 9: Bumuo ng Workflow

Parehong istruktura ng workflow tulad ng dati - kondisyonal na pag-ruta base sa availability.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## Step 10: Test Case 1 - Regular User in Paris (Walang Override)

Isang regular na user ang sumusubok mag-book sa Paris → Walang mga kwarto → Ina-redirect sa alternative_agent


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Step 11: Test Case 2 - 🌟 Priority User sa Paris (MAY Override!)

Isang priority member ang sumusubok mag-book sa Paris → Walang available na kwarto sa simula → 🌟 Middleware ang nag-override! → Dinaanan ang booking_agent

**Ito ang pangunahing demonstrasyon ng kapangyarihan ng middleware!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## Hakbang 12: Test Case 3 - Priority User sa Stockholm (Umiiral Na)

Sinusubukan ng priority user ang Stockholm → May mga bakanteng kuwarto → Hindi kailangan ng override → Dinaanan ng booking_agent

Ipinapakita nito na ang middleware ay kumikilos lamang kung kinakailangan!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## Pangunahing Mga Natutuhan at Mga Konsepto ng Middleware

### ✅ Ang Mga Natutunan Mo:

#### **1. Pattern ng Middleware na Nakabase sa Function**

Na-intercept ng middleware ang mga tawag sa function gamit ang simple async na function:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # Bago ang pagpapatupad ng function
    print("Intercepting...")
    
    # Isagawa ang function
    await next(context)
    
    # Pagkatapos ng pagpapatupad ng function - suriin ang resulta
    if context.result:
        # Baguhin ang resulta kung kinakailangan
        context.result = modified_value
```

#### **2. Pag-access ng Context at Pag-override ng Resulta**

- `context.function` - I-access ang tinatawag na function  
- `context.arguments` - Basahin ang mga argumento ng function  
- `context.kwargs` - I-access ang mga dagdag na parameter  
- `await next(context)` - Ipatupad ang function  
- `context.result` - Basahin/baguhin ang output ng function  

#### **3. Pagpapatupad ng Business Logic**

Ipinapatupad ng middleware namin ang mga benepisyo ng priority member:  
- **Regular users**: Walang pagbabago, karaniwang daloy ng trabaho  
- **Priority users**: I-override ang "no availability" → "available"  
- **Conditional logic**: Naga-override lamang kapag kinakailangan  

#### **4. Parehong Daloy ng Trabaho, Iba't Ibang Resulta**

Ang kapangyarihan ng middleware:  
- ✅ Walang pagbabago sa estruktura ng workflow  
- ✅ Walang pagbabago sa tool function  
- ✅ Walang pagbabago sa conditional routing logic  
- ✅ Middleware lang → Iba ang pag-uugali!

### 🚀 Mga Aplikasyon sa Tunay na Mundo:

1. **VIP/Premium na Mga Tampok**  
   - I-override ang mga rate limit para sa premium users  
   - Magbigay ng priority access sa mga resources  
   - I-unlock ang mga premium na tampok nang dinamiko  

2. **A/B Testing**  
   - I-route ang mga user sa iba't ibang implementasyon  
   - Subukan ang mga bagong tampok sa partikular na mga user  
   - Unti-unting pag-rollout ng mga tampok  

3. **Seguridad at Pagsunod**  
   - I-audit ang mga tawag sa function  
   - Harangan ang sensitibong operasyon  
   - Ipatupad ang mga business rules  

4. **Performance Optimization**  
   - I-cache ang mga resulta para sa partikular na mga user  
   - Laktawan ang mga magastos na operasyon kapag posible  
   - Dinamikong alokasyon ng resource  

5. **Paghawak ng Error at Retry**  
   - Hulihin at hawakan ang mga error nang maayos  
   - Ipatupad ang retry logic  
   - Fallback sa mga alternatibong implementasyon  

6. **Logging at Monitoring**  
   - Subaybayan ang oras ng pagpapatupad ng function  
   - I-log ang mga parameter at resulta  
   - I-monitor ang mga pattern ng paggamit  

### 🔑 Pangunahing Pagkakaiba sa mga Decorator:

| Tampok | Decorator | Middleware |
|---------|-----------|------------|
| **Saklaw** | Isang function lang | Lahat ng function sa agent |
| **Flexibility** | Nakapirmi sa definisyon | Dinamiko sa runtime |
| **Context** | Limitado | Buong agent context |
| **Komposisyon** | Maraming decorator | Middleware pipeline |
| **Agent-Aware** | Hindi | Oo (access sa estado ng agent) |

### 📚 Kailan Gamitin ang Middleware:

✅ **Gamitin ang middleware kapag:**  
- Kailangan mong baguhin ang pag-uugali base sa estado ng user/session  
- Gusto mong maglagay ng logic sa maraming function  
- Kailangan mong ma-access ang context ng agent  
- Ipinapatupad mo ang cross-cutting concerns (logging, auth, atbp.)  

❌ **Huwag gamitin ang middleware kapag:**  
- Simpleng input validation (gamitin ang Pydantic)  
- Specific na logic ng function (panatilihin sa function)  
- Isang beses na pagbabago (diretsong baguhin ang function)  

### 🎓 Mga Advanced na Pattern:

```python
# Maraming middleware (mahalaga ang pagkakasunod-sunod ng pagpapatupad!)
middleware=[
    logging_middleware,      # Mga log muna
    auth_middleware,         # Saka tinitingnan ang auth
    cache_middleware,        # Saka tinitingnan ang cache
    rate_limit_middleware,   # Saka rate limits
    priority_check_middleware  # Sa wakas, priority check
]

# Kondisyunal na pagpapatupad ng middleware
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # Baguhin ang resulta
    else:
        # Laktawan ang pagpapatupad nang buo
        context.result = cached_value
```

### 🔗 Mga Kaugnay na Konsepto:

- **Agent Middleware**: Nakaka-intercept ng agent.run() na mga tawag  
- **Function Middleware**: Nakaka-intercept ng mga tawag sa tool function (ang ginamit natin!)  
- **Middleware Pipeline**: Serye ng mga middleware na nagpapatupad nang sunod-sunod  
- **Context Propagation**: Pagpasa ng estado sa middleware chain  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
